In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Dataset

# 1. Define your transforms (Note the strict 64x64 resize)
train_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ColorJitter(brightness=0.2), # Safe for traffic signs
    transforms.RandomRotation(10),          # Simulates camera angle
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 2. Download the raw GTSRB dataset (no transforms applied yet)
# This will create a 'data' folder and download the files automatically
raw_dataset = datasets.GTSRB(root='./data', split='train', download=True)

# 3. Calculate the 70/15/15 split sizes
total_size = len(raw_dataset)
train_size = int(0.70 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

# 4. Perform the split with a fixed seed so Member 2 gets the exact same images
generator = torch.Generator().manual_seed(42)
train_subset, val_subset, test_subset = random_split(
    raw_dataset, [train_size, val_size, test_size], generator=generator
)

# 5. Wrapper class to safely apply augmentations ONLY to training data
class SubsetWrapper(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, index):
        image, label = self.subset[index]
        if self.transform:
            image = self.transform(image)
        return image, label
        
    def __len__(self):
        return len(self.subset)

# 6. Apply the transforms and create DataLoaders
train_data = SubsetWrapper(train_subset, transform=train_transforms)
val_data = SubsetWrapper(val_subset, transform=eval_transforms)
test_data = SubsetWrapper(test_subset, transform=eval_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"Data pipeline ready! Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

100%|██████████| 187M/187M [04:28<00:00, 699kB/s]    


Data pipeline ready! Train: 18648, Val: 3996, Test: 3996
